### Wind  Power Multilinear Regression

In this notebook, we will build multilinear regression modules to predict wind power generated from weather conditions. We will attempt to account for the day of the year and the weather conditions.
After several experiments, the best results for this approach were obtained using a simple standard scale and a ridge regression module.
 The wind speed and air density impact the wind power generated by a turbine. For this reason, we have induced wind speed and air temperature as our weather variable. Some experiments were done utilizing polynomial features or adjusting the wind speed and air temperature, but all produced worse outcomes. We will simulate predicted weather features and calculate rolling means of historical weather data after completing the relevant processing.

Our goal is to test and assess the capabilities of this relatively simple approach in comparison to the baseline module and more advanced techniques.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
from MLR_script import MlrCrossValidation

sns.set_style('whitegrid')


Here, we read the data and build the data frame that we will use to train the module.

In [10]:
df = pd.read_csv(R"..\..\1_data\final_dataframes\main_testing_dataframe.csv", index_col= 0)
df.index = pd.to_datetime(df['time'])
df_train=df[df.index.year <= 2022]
df_test=df[ df.index.year ==2023]

Best resutls where with 6 days of lagging weather forcast.

In [11]:
model = MlrCrossValidation(lag_days= 6)

In [12]:
df_clean= model.data_prep(df_train)
df_clean.head()

,Wind,wind_speed_10m_1,wind_speed_10m_33,wind_speed_10m_2,wind_speed_10m_8,wind_speed_10m_23,wind_speed_10m_5,wind_speed_10m_4,wind_speed_10m_39,wind_speed_10m_6,...,temperature_2m_27_lag5_mean,temperature_2m_28_lag5_mean,temperature_2m_29_lag5_mean,temperature_2m_30_lag5_mean,temperature_2m_32_lag5_mean,temperature_2m_31_lag5_mean,temperature_2m_34_lag5_mean,temperature_2m_38_lag5_mean,doy_sin,doy_cos
time,,,,,,,,,,,,,,,,,,,,,
2019-01-01,52554.0,18.783333,21.100000,17.650000,17.470833,24.450000,19.362500,17.341667,34.345833,17.491667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,1.000000
2019-01-02,38859.0,14.262500,7.158333,18.425000,10.841667,7.975000,10.920833,8.625000,55.729167,9.795833,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.017213,0.999852
2019-01-03,18209.0,7.300000,6.416667,4.370833,11.895833,9.150000,10.225000,11.391667,21.291667,8.445833,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.034422,0.999407
2019-01-04,31732.0,15.629167,17.675000,6.950000,19.362500,12.420833,16.183333,16.700000,33.041667,13.808333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.051620,0.998667
2019-01-05,4615.0,7.595833,8.570833,6.841667,5.820833,8.404167,5.933333,6.070833,18.720833,4.900000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.068802,0.997630


In [13]:
model.fit(df_train)
preds = model.predict(df_train)
preds.describe()

count     1455.000000
mean     31288.905155
std      10384.025264
min       6739.765939
25%      23899.290812
50%      30376.957854
75%      38124.688306
max      72918.877596
dtype: float64

In [14]:
model.cross_validate_model(df_train, k=5)

Cross-Validation Results (5-fold):
Mean MAPE: 71.49%
Mean RMSE: 14761.80
Mean R²: 0.259


In [15]:
fig=model.plot_predictions(df_train)
fig.show()

We can see above that the module capturs the overall tred in the data well but consistantly under estimates the peaks, in the data.

In [16]:
model.cross_validate_model(df_train,k=5)

Cross-Validation Results (5-fold):
Mean MAPE: 71.49%
Mean RMSE: 14761.80
Mean R²: 0.259


Comparing the models' performance against the testing data.

In [17]:
model.validate_model(df_train, df_test)

Validation MAPE: 99.83%
Validation MAE: 12414.2653
